# RAG-Based Ethical Alignment — **Kaggle** pipeline

Version Kaggle de `full_pipeline` (le code est CPU/GPU : il prend le GPU automatiquement s'il y en a un).

## Avant de lancer (panneau de droite)
1. **Settings → Accelerator → GPU T4 x2** (ou P100). Le code passe seul sur GPU → ~15-30 min au lieu de 4-8 h.
2. **Settings → Internet → On** (nécessaire pour télécharger le modèle Qwen et le dataset ETHICS).
3. **Add Input / Add Data → Upload** : envoie le **zip de ton dossier `adl_rag`** (celui qui contient `src/`, `scripts/`…). Kaggle le décompresse et le monte sous `/kaggle/input/...`.


## 1. Vérifier l'accélérateur

In [ ]:
import torch
print('CUDA dispo :', torch.cuda.is_available())
print('Device    :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Dépendances manquantes (le reste est préinstallé sur Kaggle)

In [ ]:
!pip -q install sentence-transformers faiss-cpu

## 3. Copier le code dans un dossier inscriptible
`/kaggle/input` est en lecture seule ; on copie le projet dans `/kaggle/working` (le script y écrit `outputs/`, l'index FAISS, les checkpoints).

In [ ]:
import os, glob, shutil
cands = glob.glob('/kaggle/input/**/src/run_all.py', recursive=True)
assert cands, "Projet introuvable sous /kaggle/input — as-tu ajouté le dataset (zip du dossier adl_rag) ?"
SRC = os.path.dirname(os.path.dirname(cands[0]))
DST = '/kaggle/working/adl_rag'
if os.path.exists(DST): shutil.rmtree(DST)
shutil.copytree(SRC, DST)
print('Code copié dans', DST)

In [ ]:
%cd /kaggle/working/adl_rag

### (optionnel) fp16 pour aller ~2x plus vite sur GPU
Décommente la ligne ci-dessous si tu veux maximiser la vitesse (léger écart numérique, résultats comparables). Garde fp32 si tu tiens à la consigne stricte.

In [ ]:
# !sed -i 's/TORCH_DTYPE: str = "float32"/TORCH_DTYPE: str = "float16"/' src/config.py
!grep 'TORCH_DTYPE' src/config.py | head -1

## 4. Construire l'index RAG (corpus à ~50 principes)

In [ ]:
!python -m src.rag --build_index

## 5. Smoke test (~quelques min) — vérifie que tout passe de bout en bout

In [ ]:
!python -m src.run_all --quick --all --force

In [ ]:
import json, glob
for p in sorted(glob.glob('outputs/results/*.json')):
    d = json.load(open(p)); print(f"{os.path.basename(p):20s} overall={d['overall']*100:5.1f}%")

## 6. Run complet
On vide `outputs/` pour repartir propre (sinon le script reprend les checkpoints du smoke test à n=5).

- `--all` = baseline + rag + CoT (baseline_cot, rag_cot) + ablation (rag_ablate).
- Sans `--all` : seulement baseline + rag (l'écart principal), plus rapide.

⚠️ La session GPU Kaggle est limitée à ~9 h ; le checkpointing reprend si ça coupe (relance la cellule).

In [ ]:
!rm -rf outputs/
!python -m src.run_all --all

## 7. Résultats + figures

In [ ]:
import json, os
for tag in ['baseline','rag','baseline_cot','rag_cot','rag_ablate']:
    p = f'outputs/results/{tag}.json'
    if os.path.exists(p):
        d = json.load(open(p)); print(f"{tag:14s} overall={d['overall']*100:5.1f}%")
!python scripts/make_figures.py

In [ ]:
from IPython.display import Image, display
for f in ['accuracy_bars.png','accuracy_delta.png']:
    if os.path.exists('figures/'+f): display(Image('figures/'+f))

## 8. Récupérer les résultats
Tout est dans `/kaggle/working/adl_rag/outputs/` et `/kaggle/working/adl_rag/figures/`.
Fais **Save Version** (commit) pour que `/kaggle/working` soit conservé et téléchargeable.